**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `python_prebuilt/` and the bundle's R routes (`R/lib`, see `R/README_BRIDGE.md`; the full R pipeline is the project RWDR). Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# M44 — BART-style Causal Inference
**Status: READY (ML/AI causal inference)**

---
## Formula
```
One flexible response surface f(X,D); CATE(x) = f(x,1) - f(x,0)
```
**Source**: Hill (2011), J. Computational and Graphical Statistics 20(1):217-240

---
## Validation
VALIDATED: corr 0.984, mean effect 2.021 vs true 2.004. **LIMITATION**: uses gradient boosting as the sklearn-native stand-in for BART's Bayesian sum-of-trees -- the causal logic is identical but BART's posterior uncertainty is NOT reproduced. For genuine BART use R's `bartCause`.

---
## DATA REQUIREMENTS
Panel covariates.

**To use household survey covariates**: set `C.HOUSEHOLD_CHARACTERISTICS_PATH` in `_common.py`
and add those column names to `COVARIATES` in CELL 1.

---
## How the DiD design and ML fit together
The ML stage does **not** replace the DiD design. The outcome is the pixel's **pre→post
change**, which differences out every time-invariant pixel effect exactly as DiD does; the ML
then handles confounding in the covariates and recovers *heterogeneous* effects. Read this as
"DiD identification + ML for the nuisance functions".

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (v20.55: the engine installs a missing ESTIMATOR package itself -- wheel first, then source -- and confirms 'N of N installed' at every run; these base libraries it needs before it can start)")


In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")
import _common as C      # v20.29: imported BEFORE first use (the import used to come later in this cell)

# ---- CELL 1: configuration + scenario ----
MODEL_ID = "M44"
OUTCOME  = C.current_outcome("NDVI")        # variable this run starts with; CELL 9 repeats for the rest
# ---- v20.57: THE DESIGN OF THIS RUN -- every option below is applied HERE, when this model runs, on the panel P00 built ONCE.
#      Change any of them and re-run this notebook; P00 is never re-run for it. What you set is what runs: DESIGN IN EFFECT
#      (printed below, and saved as DESIGN_IN_EFFECT.csv next to the results) lists each option, the value used and where it came from.
DESIGN_MODE    = "recommended"   # "recommended": an option set to "data" is chosen FROM THE DATA (DESIGN_RECOMMENDATION.md: export breaks,
                                 #   fill years, spillover into inner rings -- never the effect) | "manual": "data" = every ring / every year.
                                 #   A value you set (e.g. CONTROL_ZONES = "1-3", PRE_YEARS = 4) is ALWAYS used exactly as set, in both modes
TREATMENT_TIMING = "fund"        # "fund": each sub-watershed's first treated SEASON from your fund workbook -- the release timing, back-cast
                                 #   before the file's first month, the season AFTER the start (no anticipation): staggered, season-level
                                 #   cohorts (FUND_TIMING_AND_DOSE.md) | "registry": its year in data/sites/sites.csv | "fixed": TREATMENT_YEAR
TREATMENT_YEAR = 2022            # "fixed" timing -- and the fall-back for a sub-watershed the fund file / registry does not date (announced)
FUND_START_RULE = "backcast"     # fund timing: "backcast" | "share" (the amount reaches 10 % of the target) | "file_start" (a lower bound)
DOSE_VARIABLE  = "dose_intensity_per_ha"   # the dose models' dose (fund file): amount released / treatment area | "dose_amount_sws" |
                                           #   "dose_share_of_target" -- 0 on the control rings and before the treatment starts
CONTROL_ZONES  = "data"          # control ring(s): "data" | "1-5" | "1-3" | "2-5" | (1, 3, 5)
PRE_YEARS      = "data"          # years BEFORE the start: "data" | "all" | 4 (the 4 years before) | 2015 (a calendar year = the FIRST pre year)
POST_YEARS     = "data"          # years FROM the start:   "data" | "all" | 2 (the start year + 1) | 2025 (a calendar year = the LAST post year)
SEASONS        = "all"           # "all" (annual composite + Kharif/Rabi/Zaid: year AND season variation) | "seasonal" | "yearly" |
                                 #   "Rabi" | "Kharif+Rabi" | "auto" (the data decide)
EXCLUDE_TRANSITION_YEAR = False  # True = the first treated year of each series leaves the sample (robustness)
UNIT_FE        = "pixel_season"  # one fixed effect per pixel x season series (+ year x season FE) | "pixel"
SUB_WATERSHEDS = "data"          # v20.58: the sub-watershed(s) THIS run processes: "data" = every one with >= 5 % of the largest one's rows (one
                                 #   export = its own sub-watershed) | "major" (the largest only) | a name or id, e.g. "Koranahalli" | [1, 7]
FRAGMENT_RULE  = "drop"          # YOUR RULE: rows of sub-watersheds NOT processed and rows outside every polygon leave EVERY group -- treated and
                                 #   control, pre and post (SAMPLE INTEGRITY confirms it at every run) | "keep"
OVERLAP_ROWS   = "drop"          # YOUR RULE: overlapping pixels leave every group -- a pixel treated in another processed sub-watershed (as a
                                 #   control), a repeated pixel-year-season, a near-duplicate pixel, a pixel whose ring the exports disagree on | "keep"
POOLED_FE      = "site_period"   # several sub-watersheds: sub-watershed x year x season effects | "period" (one shared year x season effect)
EXCLUDE_GAPFILLED = True         # rows the exporter filled from history (GapFilled / Coverage 0) are not observations | False keeps them
OUTCOME_SCREEN    = "drop"       # v20.59: "drop" (a year-season constant across pixels -- a fill value -- or with collapsed coverage leaves the
                                 #   model; its evidence: OUTCOME_SCREEN_<outcome>.csv beside the results) | "keep" (reported and KEPT; results tagged
                                 #   _screenKept) | "off". A v20.58 run stopped here ("41 year-seasons are NOT pixel data ... Re-export it")
COVARIATES = "all"   # v20.45: THIS model's covariates -- "all" = the FOUR weather covariates (Rain, Tmax, Tmean, Tmin; tag covAll4) | "mean_temp_rain" (Tmean + Rain) | "none" | or a list. LandUse is never a covariate.
NONNEGATIVE = False   # v20.32: True = no negative value of any estimation variable enters the DiD (selection on the
                      # outcome -- NDVI/NDWI/NDMI/NDRE/SMDI are negative over water, bare soil, moisture stress);
                      # results then go to a separate "_nonneg" folder and every file records it
C.set_scenario(design_mode=DESIGN_MODE, timing=TREATMENT_TIMING, treatment_year=TREATMENT_YEAR, fund_start_rule=FUND_START_RULE,
               dose_variable=DOSE_VARIABLE, control_zones=CONTROL_ZONES, pre_years=PRE_YEARS, post_years=POST_YEARS, seasons=SEASONS,
               exclude_transition_year=EXCLUDE_TRANSITION_YEAR, unit_fe=UNIT_FE, overlap_rows=OVERLAP_ROWS, fragment_rule=FRAGMENT_RULE,
               pooled_fe=POOLED_FE, exclude_gapfilled=EXCLUDE_GAPFILLED, covariates=COVARIATES, nonnegative=NONNEGATIVE, sub_watersheds=SUB_WATERSHEDS, outcome_screen=OUTCOME_SCREEN)
# multi-site (20 sub-watersheds, SWSiD_All): run this model per site and pooled from 08_Multisite_Runs/MS01_Multisite_Runs.ipynb
SCENARIO_FROM_PANEL = True   # v20.34: what you set in THIS cell always runs; an option you leave unset (None) comes from P00's saved scenario
if SCENARIO_FROM_PANEL: C.load_scenario()
C.resolve_design()           # v20.57: timing (fund / registry / fixed), the "data" options, the fragment rule -> DESIGN IN EFFECT
CONTROL_ZONES = C.ACTIVE["control_zones"]
COVARIATES = C.DEFAULT_COVARIATES    # restored (v20.3)
RESULTS_DIR = C.results_dir(MODEL_ID)         # v20: RESULTS_ROOT/<model>/<scenario tag>
C.info(f"SCENARIO IN FORCE: {C.scenario_banner()}")
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.58")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("M44")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
C.system_report()   # v16: shows CPU / RAM / whether the A40 will be used for demeaning
C.autotune()          # v17: measures THIS machine (cores/RAM/VRAM/disk) and sets every performance knob

MAX_ROWS=None
C.ok(f"results -> {RESULTS_DIR}")
C.done(1, next_task="CELL 2 -- load the panel columns this model needs")


In [ ]:
# ---- v20.42: PRE-BUILT PACKAGE FIRST ------------------------------------------------------------------------
# A verified pre-built package (installed and checked on a known answer by P00's P12 step) computes this model's
# PRIMARY result: inside the engine function the model calls (pyfixest / wildboottest / diff-diff), or here, as
# <MODEL>_PACKAGE_<OUTCOME>.csv (diff-diff / econml / esda). The engine code below is the result only when no verified
# package exists -- and the cross-check when one does. What computes this model, and why, is printed below.
PACKAGE_RESULT = C.prebuilt_first(MODEL_ID, OUTCOME)
_eng = C.LAST_ENGINE.get(MODEL_ID, {})
print(f"[INFO]    {MODEL_ID} / {OUTCOME}: computed by {_eng.get('engine', 'engine')} -- {_eng.get('reason', '')}")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 2: load ONLY the columns this model needs ----
# v16: panel is the compact 48-column schema (pixel_id int64, outcomes float32);
# treat_core/pre_period/... aliases are regenerated by build_treatment_columns() -- nothing to change here.
C.step(2,"loading panel (column-projected)")
try:
    panel = C.load_panel(columns=C.columns_for(OUTCOME), max_rows=MAX_ROWS)
    C.ok(f"columns loaded: {list(panel.columns)}")
except Exception as e:
    C.fail(f"cannot load panel: {e}")
    C.info("Checklist:")
    C.info("  1. Run 01_Panel_Preparation (P00, or P01-P06) -- it writes did_panel_full.parquet")
    C.info("  2. v9.1 auto-discovers it via panel_location.json; if that failed, set")
    C.info("     PREPARED_PANEL in _common.py to the EXACT path P05 printed")
    raise
C.done(2, next_task="CELL 3 -- estimate M44")

In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 3: run M44 ----
C.step(3,"estimating -- BART-style Causal Inference")

def run_model():
    # v20.58: R's LONG DIFFERENCE (C.ml_long_difference -- as R's route and the Python primary: one difference per pixel x season series, the
    # season-matched outcome, pre / post by the design, the covariates' pre-period means). v20.57's engine differenced each PIXEL's raw outcome
    # (every season mixed) with the pixel's pre level among the covariates -- another estimand (M40 was 2.2 SE from R's).
    # The engine's BART stand-in (one gradient-boosted response surface on [X, D]); the headline is its effect on the TREATED series (R's
    # bartCause estimand: the ATT). It gives no posterior SD -- the headline takes the design-based SE of the same sample (said).
    df = C.build_treatment_columns(panel, control_zones=CONTROL_ZONES)
    df = df[df["in_analysis_sample"]==1]
    ld, covs = C.ml_long_difference(df, OUTCOME, COVARIATES)
    if not covs: raise C.InsufficientDataError("the ML models need covariates -- COVARIATES is empty")
    if len(ld) < 100:
        raise C.InsufficientDataError(f"only {len(ld)} series have >= 2 pre and >= 1 post rows; ML methods need >= 100")
    if ld.treat.sum() < 10 or (ld.treat == 0).sum() < 10:
        raise C.InsufficientDataError(f"need >= 10 series per arm; have {int(ld.treat.sum())} treated")
    Y = ld["dY"].values; D = ld["treat"].values.astype(float); X = ld[covs].values
    what = "pixel x season series" if C._unit_key(df, "pixel_id") != "pixel_id" else "pixels"
    C.info(f"design: {len(ld):,} {what} ({int(D.sum()):,} treated) -- one long difference each (the season-matched outcome, post mean minus "
           f"pre mean, as R's route); covariates = their pre-period means {covs}")
    cate = C.bart_style_causal(X, D, Y)
    ld["cate_bart"]=cate
    att = float(np.mean(cate[D == 1]))
    C.info(f"ATT (the stand-in's mean effect on the treated series) = {att:.6f} | CATE mean={np.mean(cate):.6f} sd={np.std(cate):.6f} "
           f"p10={np.quantile(cate,.1):.6f} p90={np.quantile(cate,.9):.6f}")
    C.info("Point estimate only -- this stand-in does not give BART's posterior intervals (R's bartCause does; the headline takes the design-based SE).")
    C.save_results({"outcome":OUTCOME,"target":"ATT","ATT_bart_style":att,"cate_mean":float(np.mean(cate)),"cate_sd":float(np.std(cate)),
                    "cate_p10":float(np.quantile(cate,.1)),"cate_p90":float(np.quantile(cate,.9)),
                    "se_note":"the gradient-boosting stand-in for BART gives no posterior SD"},
                   RESULTS_DIR, f"bart_style_{OUTCOME}.csv")
    C.save_results(ld[["unit","pixel_id","Season","site_id","cate_bart"]], RESULTS_DIR, f"bart_pixel_cate_{OUTCOME}.csv")

result=C.cell_guard(run_model)()
C.report_model_outcome(MODEL_ID, OUTCOME, result, RESULTS_DIR)   # v17.10
C.ok("M44 finished") if result is not None else C.warn("M44 produced no result -- see message above")

# ---- MEMORY RELEASE (v14): free this model's frames so the next one starts clean ----
try:
    C.release(*[v for k,v in list(globals().items()) if k in ("panel","df","result") and v is not None])
except Exception as _e:
    print(f"[INFO]    release skipped: {_e}")
C.done(3, next_task="any model -- see MODEL_GUIDE_CATEGORY_WISE.md; recommended sequence starts at M01")

---
## Interpreting these results
- **Mean CATE** should be broadly consistent with the average effect from M01/M27. A large gap
  means either strong heterogeneity or a specification problem — check both.
- **p10 vs p90 spread** is the heterogeneity signal: a wide spread means the programme worked
  very differently in different places.
- **Feature importance** (M43) tells you *which* characteristic drives that difference — the
  most actionable output in this whole pipeline for targeting future watershed investment.

In [ ]:
# ---- CELL 9: run the REMAINING outcome variables automatically (v17.10) ----
# Run All on this notebook now produces every variable in C.SELECTED_OUTCOMES, one after another,
# with no manual edits. Set AUTO_NEXT_OUTCOMES = False to estimate only the outcome chosen in CELL 1.
AUTO_NEXT_OUTCOMES = True
OVERWRITE_EXISTING = False        # True = recompute outcomes that already have result files
if AUTO_NEXT_OUTCOMES:
    C.run_other_outcomes("M44", first_outcome=OUTCOME, overwrite=OVERWRITE_EXISTING)
